In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from qiskit import qasm2
from qiskit.qasm2 import LEGACY_CUSTOM_INSTRUCTIONS
from stable_baselines3 import A2C, DQN, PPO

from data_processing import compute_metric_scale, OPTIMIZER_NAMES
from environment import OptimizerChainEnv


In [ ]:
# ------------------------------------------------------------
# Edit these values.
# ------------------------------------------------------------

repo_root = Path.cwd().resolve()
while not (repo_root / "dbs" / "combined_master.csv").exists():
    if repo_root.parent == repo_root:
        raise FileNotFoundError("Could not locate project root.")
    repo_root = repo_root.parent
v4_dir = repo_root / "machine_learning" / "RL" / "two_opt_chain_rl_v4"

model_path = v4_dir / "models" / "optimizer_chain_ppo_mixed_weighted_seed42_fbplus_all_features_rw0p1_tqw1p0_dw0p1_frw0p5.zip"
algorithm = "ppo"   # "ppo", "a2c", or "dqn"

train_set = pd.read_csv(v4_dir / "splits" / "train.csv")
test_set = pd.read_csv(v4_dir / "splits" / "test.csv")

# Pick one row from the test set.
x = 123
circuit = test_set.iloc[x]



In [ ]:
test_set

In [ ]:
# ------------------------------------------------------------
# Load the model and build the v4 environment.
# This uses the existing environment.
# ------------------------------------------------------------

algorithm_classes = {
    "ppo": PPO,
    "a2c": A2C,
    "dqn": DQN,
}

# The environment needs category order and metric scaling.
category_names = sorted(pd.concat([train_set["original_category"], test_set["original_category"]]).dropna().unique())
metric_scale = compute_metric_scale(train_set)

# Load the saved RL model.
model = algorithm_classes[algorithm.lower()].load(str(model_path))

# Build the environment on the full test split.
env = OptimizerChainEnv(
    frame=test_set,
    optimizer_names=OPTIMIZER_NAMES,
    category_names=category_names,
    metric_scale=metric_scale,
    shuffle_each_epoch=False,
)

# Identify the chosen original circuit.
chosen_circuit_name = circuit["original_circuit"]
circuit_rows = test_set[test_set["original_circuit"] == chosen_circuit_name].copy()
original_row = circuit_rows.iloc[0]

print("Model:", model_path)
print("Chosen circuit:", chosen_circuit_name)
print("Category:", original_row["original_category"])
print("Original path:", original_row["original_circuit_path"])


In [ ]:
# ------------------------------------------------------------
# Run the model on the chosen circuit through the v2 environment.
# ------------------------------------------------------------

# Reset the environment to this specific circuit.
observation, info = env.reset(options={"circuit": chosen_circuit_name})

# First model decision.
action_1, _ = model.predict(observation, deterministic=True)
action_1 = int(action_1)
observation, reward_1, done_1, truncated_1, info_1 = env.step(action_1)

if info_1["selected_chain"] is None:
    raise ValueError("The first action was invalid for this circuit.")

chain_after_step_1 = info_1["selected_chain"]

# Second model decision.
action_2, _ = model.predict(observation, deterministic=True)
action_2 = int(action_2)
observation, reward_2, done_2, truncated_2, info_2 = env.step(action_2)

final_chain = info_2["selected_chain"]

print("Step 1 chain:", chain_after_step_1)
print("Step 2 / final chain:", final_chain)
print("Reward after step 1:", reward_1)
print("Reward after step 2:", reward_2)


In [ ]:
# ------------------------------------------------------------
# Show the original circuit, the circuit after step 1,
# and the final circuit after step 2.
# ------------------------------------------------------------

def load_circuit(path_text):
    return qasm2.load(str(path_text), custom_instructions=LEGACY_CUSTOM_INSTRUCTIONS)

def initial_features(row):
    return pd.DataFrame(
        {
            "value": [
                row["initial_num_qubits"],
                row["initial_depth"],
                row["initial_two_qubit_gates"],
                row["initial_total_gates"],
                row["initial_two_qubit_depth"],
                row["initial_gate_density"],
                row["initial_two_qubit_ratio"],
            ]
        },
        index=[
            "num_qubits",
            "depth",
            "two_qubit_gates",
            "total_gates",
            "two_qubit_depth",
            "gate_density",
            "two_qubit_ratio",
        ],
    )

def final_features(row):
    return pd.DataFrame(
        {
            "value": [
                row["final_num_qubits"],
                row["final_depth"],
                row["final_two_qubit_gates"],
                row["final_total_gates"],
                row["final_two_qubit_depth"],
                0 if row["final_num_qubits"] == 0 else row["final_total_gates"] / row["final_num_qubits"],
                0 if row["final_total_gates"] == 0 else row["final_two_qubit_gates"] / row["final_total_gates"],
            ]
        },
        index=[
            "num_qubits",
            "depth",
            "two_qubit_gates",
            "total_gates",
            "two_qubit_depth",
            "gate_density",
            "two_qubit_ratio",
        ],
    )

def show_circuit_stage(title, circuit_obj, feature_df):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)
    display(feature_df)
    #try:
    print(circuit_obj.draw(output="text"))
        #plt.show()
        #plt.close(figure)
    #except Exception:
        #print(circuit_obj.draw(output="text", fold=100))

# Original circuit.
original_circuit = load_circuit(original_row["original_circuit_path"])
show_circuit_stage("Original circuit", original_circuit, initial_features(original_row))

# Circuit after optimizer 1.
row_after_step_1 = circuit_rows[circuit_rows["chain_name"] == chain_after_step_1].iloc[0]
circuit_after_step_1 = load_circuit(row_after_step_1["artifact_path"])
show_circuit_stage(
    f"Circuit after step 1: {chain_after_step_1}",
    circuit_after_step_1,
    final_features(row_after_step_1),
)

# Final circuit after step 2.
row_final = circuit_rows[circuit_rows["chain_name"] == final_chain].iloc[0]
final_circuit = load_circuit(row_final["artifact_path"])
show_circuit_stage(
    f"Final circuit: {final_chain}",
    final_circuit,
    final_features(row_final),
)
